# 11 추가 사진 채점 (03b로 더 받은 터널 입구 사진)
| 항목 | 내용 |
|---|---|
| 학습 | 71786 3,624장 (8~9월, 입구 1,927 + 터널 없는 고속도로 1,697). ②a = Mapillary 사진은 학습에 안 씀. 10과 같은 대표 모델 · 해 높이 규칙 |
| 새 시험 | 03b로 받은 사진 중 '입구'로 분류한 새 사진 (사진분류_재혁2.csv, 기존 172장에 없는 것만) |
| 정답 | 04와 같음: 천리안 일사량(이 노트북에서 받음, 캐시 data/gk2a_photo 같이 씀) → 바깥 밝기 → 입구 휘도비 → 상/중/하 |
| 구름 | 가장 가까운 ASOS 지점 전운량 (sunsight_data/data/1_원본/asos_hourly_all.csv, 2023~2025만 있음 → 그 밖은 빈칸 = 학습 중앙값) |
| 채점 | ① 고속도로 입구 → 기존 66장에 더해 주 판정 다시 ② 일반 도로터널 입구 → "같은 공식이라면" 참고 줄 (일반 도로터널 설정값 280은 100km/h 지침값이라 실제와 다를 수 있음) |
| 주의 | 천리안 2A는 2019년 7월부터 자료 → 그 전 사진은 정답을 못 만듦(빠짐). 실행하려면 sunsight_data/.env 의 KMA_APIHUB_KEY 필요 (키 값은 출력 안 함) |

In [ ]:
import os, time, math, json
from pathlib import Path
import numpy as np, pandas as pd, requests
DATA = Path('data'); SD = Path('..') / 'sunsight_data'
CACHE = DATA / 'gk2a_photo'; CACHE.mkdir(parents=True, exist_ok=True)
BASE_URL = 'https://apihub.kma.go.kr/api/typ01/cgi-bin/url/nph_sun_sat_ana_txt'
def env(key):
    p = SD / '.env'
    if p.exists():
        for line in p.read_text(encoding='utf-8-sig').splitlines():
            if line.strip().startswith(key + '='): return line.split('=', 1)[1].strip().strip('"').strip("'")
    return os.environ.get(key)
AUTH_KEY = env('KMA_APIHUB_KEY'); print('천리안 키', '있음' if AUTH_KEY else '없음 (캐시에 있는 것만 씀)')
# 07의 공통 함수(정답 공식 · 등급 · 해 높이 · 사진 숫자 · 규칙)를 그대로 가져옴 → 같은 자
nb7 = json.load(open('07_밝기추정_규칙vsML.ipynb', encoding='utf-8'))
src = [''.join(c['source']) for c in nb7['cells']]
exec([s for s in src if 'def photo_features' in s][0])
exec([s for s in src if 'class Rule1Hand' in s][0].replace('import lightgbm as lgb\n', 'pass\n'))
PLUS = FEATURES + ['sun_alt', 'cloud']; LAB = ['하', '중', '상']
def gr(y, cls): return grade(ratio_from_L20(10 ** np.asarray(y), np.asarray(cls)))

## 1. 새 입구 사진 고르기

In [ ]:
meta = pd.read_csv(DATA / 'tunnel_photos_추가_meta.csv', dtype={'image_id': str})
meta['captured_at'] = pd.to_datetime(meta.captured_at.astype(str).str[:19])
old = pd.read_csv(DATA / 'tunnel_photos_meta.csv', dtype={'image_id': str})
cl2 = pd.read_csv(DATA / '사진분류_재혁2.csv', dtype=str, encoding='utf-8-sig')[['image_id', 'class']]
new = meta.drop_duplicates('image_id').merge(cl2, on='image_id', how='inner')
new = new[~new.image_id.isin(old.image_id)]                         # 기존 172장은 뺌 (이미 10에서 채점)
print('새 사진', len(new), '장 | 분류', new.groupby(['class', '도로']).size().to_dict())
ph = new[new['class'] == '입구'].copy(); ph['cls'] = '입구'
print('새 입구', len(ph), '장 | 도로', ph['도로'].value_counts().to_dict())

## 2. 천리안 정답 받기 (04와 같은 방법)

In [ ]:
ph['t0'] = ph.captured_at.dt.floor('h'); ph['lat_r'], ph['lon_r'] = ph.lat.round(2), ph.lon.round(2)
ph['file'] = [f'{a:.2f}_{b:.2f}_{t:%Y%m%d%H}.txt' for a, b, t in zip(ph.lat_r, ph.lon_r, ph.t0)]
utc = lambda x: (x - pd.Timedelta(hours=9)).strftime('%Y%m%d%H%M')
ph['천리안 자료 기간'] = ph.captured_at >= pd.Timestamp('2019-07-25')
for r in ph[ph['천리안 자료 기간']].drop_duplicates('file').itertuples():
    f = CACHE / r.file
    if (f.exists() and f.stat().st_size > 0) or not AUTH_KEY: continue
    try:
        res = requests.get(BASE_URL, timeout=60, params={'int': '30', 'tm1': utc(r.t0 - pd.Timedelta(hours=1)), 'tm2': utc(r.t0 + pd.Timedelta(hours=2)),
                                                        'lat': f'{r.lat_r:.4f}', 'lon': f'{r.lon_r:.4f}', 'authKey': AUTH_KEY})
        txt = res.text
        if res.status_code != 200 or 'error' in txt[:200].lower() or '인증' in txt[:200]: print('실패', r.file, res.status_code, txt[:80].replace(AUTH_KEY, '***')); continue
        f.write_text(txt, encoding='utf-8'); time.sleep(0.2)
    except Exception as e: print('실패', r.file, type(e).__name__, str(e)[:80].replace(AUTH_KEY, '***'))
def parse(txt):
    lines = [l for l in txt.splitlines() if l.strip().startswith('|')]
    if not lines: return []
    head = [x.strip() for x in lines[0].strip('|').split('|')]; out = []
    for l in lines[1:]:
        d = dict(zip(head, [x.strip() for x in l.strip('|').split('|')]))
        for k in head[4:]: out.append({'time_kst': pd.to_datetime(k, format='%Y%m%d%H%M') + pd.Timedelta(hours=9), 'gk2a_icsr': pd.to_numeric(d[k], errors='coerce')})
    return out
g = []
for f in ph.file.unique():
    p = CACHE / f
    if p.exists():
        for x in parse(p.read_text(encoding='utf-8')): x['file'] = f; g.append(x)
g = pd.DataFrame(g, columns=['time_kst', 'gk2a_icsr', 'file']).dropna().drop_duplicates(['file', 'time_kst'])
ph['tm'] = ph.captured_at.dt.ceil('h')                                 # 04·07과 같게: 다음 정시 1시간 값
ph = ph.merge(g, left_on=['file', 'tm'], right_on=['file', 'time_kst'], how='left').drop(columns='time_kst')
ph['E'] = ph.gk2a_icsr.where(ph['천리안 자료 기간'])            # 2019-07 전은 천리안 자료 없음
print('천리안 정답 있음', int(ph.E.notna().sum()), '/', len(ph), '| 2019-07 전이라 못 만듦', int((~ph['천리안 자료 기간']).sum()))

## 3. 구름 · 사진 숫자 · 해 높이

In [ ]:
st = pd.read_csv(SD / 'data' / '1_원본' / 'asos_stations.csv', encoding='utf-8-sig').rename(columns={'지점': 'stn_id', '위도': 'slat', '경도': 'slon'})
aso = pd.read_csv(SD / 'data' / '1_원본' / 'asos_hourly_all.csv', usecols=['stn_id', 'tm', 'icsr', 'dc10Tca'], parse_dates=['tm'])
meas = aso.groupby('stn_id').icsr.max(); st = st[st.stn_id.isin(meas[meas > 0].index)].reset_index(drop=True)
def near(lat, lon):
    d = np.hypot(st.slat - lat, (st.slon - lon) * math.cos(math.radians(lat))); return st.stn_id[d.idxmin()]
ph['stn_id'] = [near(a, b) for a, b in zip(ph.lat, ph.lon)]
ph = ph.merge(aso[['stn_id', 'tm', 'dc10Tca']].drop_duplicates(['stn_id', 'tm']), on=['stn_id', 'tm'], how='left').rename(columns={'dc10Tca': 'cloud'})
rows = []
for r in ph.itertuples():
    f = photo_features(DATA / 'tunnel_photos_추가' / f'{r.image_id}.jpg')
    if f: f['image_id'] = r.image_id; rows.append(f)
ph = ph.merge(pd.DataFrame(rows), on='image_id', how='left')
ph['sun_alt'] = sun_altitude(ph.captured_at, ph.lat, ph.lon); ph['month'] = ph.captured_at.dt.month
print('구름 있음', int(ph.cloud.notna().sum()), '/', len(ph), '| 사진 숫자 있음', int(ph.sky_mean.notna().sum()), '/', len(ph))
sc = ph[ph.E > 0.05].copy(); sc['y'] = np.log10(L20_from_E(sc.E))
print('채점 가능 (정답 있음 · 해 있음)', len(sc), '장 | 도로', sc['도로'].value_counts().to_dict())
sc[['image_id', '도로', 'tunnel', 'captured_at', 'month', 'sun_alt', 'cloud', 'E']]

## 4. 대표 모델 · 해 높이 규칙 (10과 같은 학습: 71786만, C = 0.01)

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score
f1 = lambda a, b: f1_score(a, b, average='macro', labels=LAB, zero_division=0)
X = lambda d, c: d[c].to_numpy(float)
tr = pd.read_csv(DATA / 'aihub_71786' / '학습표_71786.csv')
mp = pd.read_csv(DATA / '시험표_mapillary.csv', dtype={'image_id': str}, parse_dates=['captured_at'])
test = mp[mp.test_ok & (mp.cls == '입구')].reset_index(drop=True)
logit = lambda C: (lambda: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), LogisticRegression(C=C, class_weight='balanced', max_iter=2000)))
stump = lambda: make_pipeline(SimpleImputer(strategy='median'), DecisionTreeClassifier(max_depth=1, class_weight='balanced', random_state=0))
ridge = lambda: make_pipeline(SimpleImputer(strategy='median'), StandardScaler(), Ridge(alpha=10))
def two_stage(make, cols):
    def fp(a, b):
        ga = gr(a.y, a.cls); s2 = ga != '하'
        m1 = make().fit(X(a, cols), ga != '하'); m2 = make().fit(X(a, cols)[s2], ga[s2] == '상')
        r1 = m1.predict(X(b, cols)).astype(bool); r2 = m2.predict(X(b, cols)).astype(bool)
        return np.where(~r1, '하', np.where(r2, '상', '중'))
    return fp
def add_est(a, b):
    ea = np.empty(len(a))
    for x, y in GroupKFold(5).split(a, groups=a.scene.values): ea[y] = ridge().fit(a.iloc[x][PLUS], a.y.iloc[x]).predict(a.iloc[y][PLUS])
    a = a.copy(); b = b.copy(); a['est_L'] = ea; b['est_L'] = ridge().fit(a[PLUS], a.y).predict(b[PLUS]); return a, b
def main_model(a, b):
    a2, b2 = add_est(a, b); return two_stage(logit(0.01), PLUS + ['est_L'])(a2, b2)
rule = two_stage(stump, ['sun_alt'])
NAME_M, NAME_R = '대표 모델 (추정 밝기 + 2단계 로지스틱)', '해 높이 규칙 (2단계 경계)'
if len(sc):
    sc[NAME_M] = main_model(tr, sc); sc[NAME_R] = rule(tr, sc); sc['정답 등급'] = gr(sc.y, sc.cls)
    sc['추정 L20 (대표 모델)'] = (10 ** add_est(tr, sc)[1].est_L).round(1); sc['정답 L20'] = (10 ** sc.y).round(1)
P66 = {NAME_M: main_model(tr, test), NAME_R: rule(tr, test)}; g66 = gr(test.y, test.cls)
print('기존 입구 66장 다시 계산: 대표', round(f1(g66, P66[NAME_M]), 3), '| 규칙', round(f1(g66, P66[NAME_R]), 3), '(10과 같으면 .844 · .789)')

## 5. 채점

In [ ]:
def row(nm, g, a, b):
    A, B = a == g, b == g
    return {'시험 사진': nm, '장수': len(g), '정답 등급': pd.Series(g).value_counts().reindex(LAB, fill_value=0).to_dict(),
            '대표 모델 F1': round(f1(g, a), 3) if len(g) else None, '해 높이 규칙 F1': round(f1(g, b), 3) if len(g) else None,
            '대표 정확도': round(float(A.mean()), 2) if len(g) else None, '규칙 정확도': round(float(B.mean()), 2) if len(g) else None,
            '둘 다 맞힘': int((A & B).sum()), '대표 모델만 맞힘': int((A & ~B).sum()), '규칙만 맞힘': int((~A & B).sum()), '둘 다 틀림': int((~A & ~B).sum())}
hw = sc[sc['도로'] == '고속도로'] if len(sc) else sc; lo = sc[sc['도로'] != '고속도로'] if len(sc) else sc
g67 = np.concatenate([g66, hw['정답 등급'].values]) if len(hw) else g66
R = [row('기존 입구 66장', g66, P66[NAME_M], P66[NAME_R]),
     row('① 주 판정: 66장 + 새 고속도로 입구', g67, np.concatenate([P66[NAME_M], hw[NAME_M].values]) if len(hw) else P66[NAME_M],
         np.concatenate([P66[NAME_R], hw[NAME_R].values]) if len(hw) else P66[NAME_R]),
     row('② 참고: 새 일반 도로터널 입구 ("같은 공식이라면")', lo['정답 등급'].values if len(lo) else np.array([]),
         lo[NAME_M].values if len(lo) else np.array([]), lo[NAME_R].values if len(lo) else np.array([]))]
res = pd.DataFrame(R); res.to_csv(DATA / '추가사진_채점.csv', index=False, encoding='utf-8-sig'); display(res)
if len(sc):
    keep = ['image_id', '도로', 'tunnel', 'captured_at', 'month', 'sun_alt', 'cloud', 'E', '정답 L20', '추정 L20 (대표 모델)', '정답 등급', NAME_M, NAME_R]
    sc[keep].to_csv(DATA / '추가사진_사진별.csv', index=False, encoding='utf-8-sig'); display(sc[keep])
print('저장: data/추가사진_채점.csv · 추가사진_사진별.csv (장 수가 작아 다시 뽑기 범위는 안 냄)')